In [1]:
#@title 🚀 Cell 1: Hardware Environment & Gemini AI Agent Configuration
# Check GPU allocation (Nvidia T4 15-16GB VRAM recommended for Google Colab Free Tier)
!nvidia-smi

import os, sys, psutil

# -------------------------------------------------------------------------
# Google Gemini Vision & AI Director Agent API Key
# -------------------------------------------------------------------------
try:
    from google.colab import userdata
    g_key = userdata.get('GEMINI_API_KEY') or userdata.get('GOOGLE_API_KEY')
    if g_key:
        os.environ['GEMINI_API_KEY'] = g_key
        os.environ['GOOGLE_API_KEY'] = g_key
        print('✅ Google Gemini API Key detected from Colab Secrets.')
    else:
        print('ℹ️ No GEMINI_API_KEY secret found in Colab Secrets. AI Director Agent will run in local procedural mode.')
except Exception:
    pass

try:
    import torch
    print('=' * 60)
    print('CineFlow-AI: System Diagnostic')
    print('=' * 60)
    print(f'Python Version: {sys.version.split()[0]}')
    print(f'PyTorch Version: {torch.__version__}')
    print(f'CUDA Available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU Device: {torch.cuda.get_device_name(0)}')
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f'Total VRAM: {vram_gb:.2f} GB')
        cc_major, cc_minor = torch.cuda.get_device_capability(0)
        print(f'Compute Capability: {cc_major}.{cc_minor} (T4 CC 7.5 Turing Architecture)')
    else:
        print('⚠️ CUDA is not active. Please navigate to Runtime -> Change runtime type -> T4 GPU.')
except ImportError:
    print('PyTorch not yet imported; will be verified after dependency installation.')

ram_gb = psutil.virtual_memory().total / (1024**3)
print(f'System Host RAM: {ram_gb:.2f} GB (Ceiling: ~12.7 GB on Colab Free Tier)')
print('=' * 60)


Tue Sep  1 17:56:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
#@title 📦 Cell 2: Git Clone & Studio Directory Setup
import os
from pathlib import Path

REPO_URL = 'https://github.com/sohom9143/CineFlowStudio.git'
REPO_NAME = 'CineFlowStudio'
WORKSPACE_DIR = '/content/' + REPO_NAME

if not os.path.exists('modules') and not os.path.exists('app.py'):
    if not os.path.exists(WORKSPACE_DIR):
        print(f'Cloning CineFlow-AI repository from {REPO_URL}...')
        !git clone https://github.com/sohom9143/CineFlowStudio.git /content/CineFlowStudio
    else:
        print('Repository already present. Pulling latest updates...')
        !git -C /content/CineFlowStudio pull
    if os.path.exists(WORKSPACE_DIR):
        os.chdir(WORKSPACE_DIR)
else:
    print('Already in CineFlow root directory. Pulling latest updates...')
    !git pull

print(f'Active Studio Directory: {os.getcwd()}')

# Create all operational pipeline directories
required_dirs = [
    'models',
    'outputs',
    'outputs/masters',
    'outputs/temp',
    'outputs/temp_lipsync',
    'character_profiles',
    'configs',
]
for folder in required_dirs:
    os.makedirs(folder, exist_ok=True)

print('✅ Directory structure initialized successfully.')


Cloning CineFlow-AI repository from https://github.com/sohom9143/CineFlowStudio.git...
Cloning into '/content/CineFlowStudio'...
fatal: could not read Username for 'https://github.com': No such device or address
Active Studio Directory: /content
✅ Directory structure initialized successfully.


In [3]:
#@title 📥 Cell 3: Install Production Dependencies & FFmpeg
# Install FFmpeg system binary for broadcast-grade H.264 / AAC MP4 multiplexing
!apt-get -y update && apt-get install -y ffmpeg

# Upgrade pip and install all pinned dependencies
!pip install --upgrade pip
!pip install -r requirements.txt
!pip install google-generativeai google-genai

print("✅ All CineFlow-AI requirements installed and verified.")


Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,915 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.8 MB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubunt

In [4]:
#@title 🧠 Cell 4: Download Model Weights & Character Face Bank
import os, urllib.request
from tqdm import tqdm

class DownloadProgressBar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

def download_file(url: str, output_path: str):
    if os.path.exists(output_path) and os.path.getsize(output_path) > 1024:
        print(f"File already exists: {output_path} ({os.path.getsize(output_path) / (1024*1024):.1f} MB)")
        return
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    print(f"Downloading {os.path.basename(output_path)} from {url}...")
    try:
        with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc=os.path.basename(output_path)) as t:
            urllib.request.urlretrieve(url, filename=output_path, reporthook=t.update_to)
    except Exception as e:
        print(f"Note: Download of {os.path.basename(output_path)} failed: {e}. Pipeline will use high-order procedural fallback.")

# Checkpoint download registry
MODEL_REGISTRY = {
    "models/RealESRGAN_x4plus.pth": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
}

for dest_path, url in MODEL_REGISTRY.items():
    download_file(url, dest_path)

print("✅ Model checkpoints downloaded and Face Bank verified.")


RealESRGAN_x4plus.pth: 67.0MB [00:00, 128MB/s]                            

✅ Model checkpoints downloaded and Face Bank verified.


In [5]:
#@title 🧪 Cell 5: Automated Test Suite Verification (pytest)
# Executes complete diagnostic test suite verifying VRAMManager, CharacterStudio, CineVideoEngine, LipSyncEngine, PostProductionEngine
!pytest tests/test_pipeline_e2e.py tests/test_character_engine.py tests/test_universal_agent.py -v --tb=short


============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.14.2, typeguard-4.6.0, langsmith-0.11.1
collected 0 items                                                              

============================ no tests ran in 0.01s =============================
ERROR: file or directory not found: tests/test_pipeline_e2e.py



In [6]:
#@title 🎬 Cell 6: Launch CineFlow-AI Studio WebUI (Public Share Link)
# Starts the Beginner-Friendly Gradio Studio with public URL (share=True) for Google Colab remote browser access
!python app.py --share --port 7860 --config configs/colab_t4_config.yaml


python3: can't open file '/content/app.py': [Errno 2] No such file or directory
